In [96]:
import math
import pickle
import warnings
import torch
import matplotlib.pyplot as plt
from bayes_opt import BayesianOptimization
from utils import *
from model import TranSiGen
from dataset import TranSiGenDataset

In [97]:
warnings.filterwarnings('ignore')

In [98]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [99]:
dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Carregamento dos dados

Os dados são carregados a partir de um arquivo `.h5`:

In [100]:
data_path = '../data/LINCS2020/data_example/processed_data_id.h5'

In [101]:
data = load_from_HDF(data_path)

In [102]:
cell_count = len(set(data['cid']))
feat_type = 'KPGT'
batch_size = 64
learning_rate = 1e-3
beta = 0.1
dropout = 0.1
weight_decay = 1e-5
n_folds = 5
random_seed = 364039
split_type = 'smiles_split'
features_dim = 2304
features_embed_dim = [400]
n_latent = 100
# init_mode = 'pretrain_shRNA'
init_mode = 'random'
n_epochs = 300
molecule_path = '../data/LINCS2020/idx2smi.pickle'

In [103]:
local_out = './results/trained_models_{}_cell_{}/{}/feature_{}_init_{}/'.format(cell_count, split_type, random_seed, feat_type, init_mode)

In [104]:
with open(molecule_path, 'rb') as f:
    idx2smi = pickle.load(f)

## Divisão de treino e teste

In [105]:
pair, pairv, pairt = split_data(data, n_folds=n_folds, split_type=split_type, rnds=random_seed)

In [106]:
train = TranSiGenDataset(
    LINCS_index=pair['LINCS_index'],
    mol_feature_type=feat_type,
    mol_id=pair['canonical_smiles'],
    cid=pair['cid']
)

valid = TranSiGenDataset(
    LINCS_index=pairv['LINCS_index'],
    mol_feature_type=feat_type,
    mol_id=pairv['canonical_smiles'],
    cid=pairv['cid']
)

test = TranSiGenDataset(
    LINCS_index=pairt['LINCS_index'],
    mol_feature_type=feat_type,
    mol_id=pairt['canonical_smiles'],
    cid=pairt['cid']
)

train_loader = torch.utils.data.DataLoader(dataset=train, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=4, worker_init_fn=seed_worker)
valid_loader = torch.utils.data.DataLoader(dataset=valid, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=4, worker_init_fn=seed_worker)
test_loader = torch.utils.data.DataLoader(dataset=test, batch_size=batch_size, shuffle=True, drop_last=False, num_workers=4, worker_init_fn=seed_worker)

In [107]:
print("Tamanho do Dataset de treino:", len(train))
print("Tamanho do Dataset de validação:", len(valid))
print("Tamanho do Dataset de teste:", len(test))

Tamanho do Dataset de treino: 65
Tamanho do Dataset de validação: 18
Tamanho do Dataset de teste: 17


In [122]:
print(train_loader)

## Criação do Modelo

In [110]:
pbounds = {
    'n_latent': (10, 600), # dimensão do espaço latente Z1, Z2
    'dropout': (0, 0.9),   # fator de dropout aplicado às redes de enc e dec
    'beta': (0, 1),        # hiperparâmetro beta que multiplica o termo KL na loss do VAE
    'features_embed_dim': (100, 1000) # dimensão do embedding da molécula dentro da rede
}

In [111]:
def train_model(n_latent, dropout, beta, features_embed_dim, is_best=False):
    path_model = f'./results/nl={round(n_latent)}-dp={dropout:.4f}-bt={beta:.4f}-fd={round(features_embed_dim)}--'
    if is_best:
        path_model = './results/'
    model = TranSiGen(
        n_genes=978,
        n_latent=round(n_latent),
        n_en_hidden=[1200],
        n_de_hidden=[800],
        features_dim=features_dim,
        features_embed_dim=[round(features_embed_dim)],
        init_w=True,
        beta=beta,
        device=dev,
        dropout=dropout,
        path_model=path_model,
        random_seed=random_seed
    ).to(dev)
    
    epoch_hist, best_epoch = model.train_model(
        train_loader=train_loader,
        test_loader=valid_loader,
        n_epochs=400,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        save_model=True,
        verbose=True
    )
    
    return -epoch_hist['valid_loss'][best_epoch]

In [25]:
optimizer = BayesianOptimization(
    f=train_model,
    pbounds=pbounds,
    random_state=random_seed,
)

In [ ]:
optimizer.maximize(init_points=0, n_iter=20)

In [114]:
model = TranSiGen(
    n_genes=978,
    n_latent=80,
    n_en_hidden=[1200],
    n_de_hidden=[800],
    features_dim=features_dim,
    features_embed_dim=[173],
    init_w=True,
    beta=0.6542363348175738,
    device=dev,
    dropout=0.08043982573185449,
    path_model='',
    random_seed=random_seed
).to(dev)

In [ ]:
train_model(**optimizer.max['params'], is_best=True)

In [ ]:
model = torch.load('results/best_model.pt')

## Avaliação do Modelo no Conjunto Teste

In [ ]:
_, _, test_metrics_dict_ls = model.test_model(loader=test_loader, metrics_func=['pearson'])

for name, rec_dict_value in zip(['test'], [test_metrics_dict_ls]):
    df_rec = pd.DataFrame.from_dict(rec_dict_value)
    smi_ls = []
    for smi_id in df_rec['cp_id']:
        smi_ls.append(idx2smi[smi_id])
    df_rec['canonical_smiles'] = smi_ls

In [ ]:
df_rec

,x1_rec_pearson,x2_rec_pearson,x2_pred_pearson,DEG_rec_pearson,DEG_pred_pearson,cp_id,cid,sig,canonical_smiles
0,0.944225,0.926119,0.922537,0.199172,0.166481,6,A375,A375_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
1,0.844968,0.853367,0.849935,0.163265,0.171045,6,HT29,HT29_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
2,0.942965,0.918368,0.897666,0.295777,0.269045,9,A549,A549_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
3,0.977116,0.955755,0.947049,0.571797,0.529164,13,ASC,ASC_13,Brc1ccc(NC(=O)N2NC(=O)[C@H]([C@@H]2c2ccccc2)c2...
4,0.972269,0.954420,0.945741,0.421421,0.303794,9,PC3,PC3_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
5,0.986818,0.971507,0.966324,0.396427,0.387523,13,HA1E,HA1E_13,Brc1ccc(NC(=O)N2NC(=O)[C@H]([C@@H]2c2ccccc2)c2...
6,0.957665,0.950882,0.941815,0.342316,0.271578,6,HA1E,HA1E_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
7,0.942791,0.963365,0.952109,0.578217,0.486590,13,A549,A549_13,Brc1ccc(NC(=O)N2NC(=O)[C@H]([C@@H]2c2ccccc2)c2...
8,0.911938,0.942710,0.932689,0.604693,0.549577,9,VCAP,VCAP_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
9,0.942084,0.927439,0.921694,0.175447,0.146525,6,A549,A549_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1


# Regression 

In [143]:
import torch
import numpy as np
import pandas as pd
from sklearn.linear_model import BayesianRidge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

def extract_data(dataset):
    x_list = []
    y_list = []
    molfeat_list = []
    cp_id_list = []
    cid_list = []
    sig_list = []
    
    for i in range(len(dataset)):
        x_data, y_data, mol_features, cp_id, cid, sig = dataset[i]
        
        if isinstance(x_data, torch.Tensor):
            x_data = x_data.detach().cpu().numpy()
        if isinstance(y_data, torch.Tensor):
            y_data = y_data.detach().cpu().numpy()
        if isinstance(mol_features, torch.Tensor):
            mol_features = mol_features.detach().cpu().numpy()
        
        x_list.append(x_data)
        y_list.append(y_data)
        molfeat_list.append(mol_features)
        
        cp_id_list.append(cp_id)
        cid_list.append(cid)
        sig_list.append(sig)
    
    return x_list, y_list, molfeat_list, cp_id_list, cid_list, sig_list


x_train, y_train, molfeat_train, cp_id_train, cid_train, sig_train = extract_data(train)
x_valid, y_valid, molfeat_valid, cp_id_valid, cid_valid, sig_valid = extract_data(valid)
x_test,  y_test,  molfeat_test,  cp_id_test,  cid_test,  sig_test  = extract_data(test)

x_train = np.array(x_train)
y_train = np.array(y_train)
x_valid = np.array(x_valid)
y_valid = np.array(y_valid)
x_test  = np.array(x_test)
y_test  = np.array(y_test)


In [ ]:

model = MultiOutputRegressor(BayesianRidge())
model.fit(x_train, y_train)


MultiOutputRegressor(estimator=BayesianRidge())

In [ ]:

y_pred_valid = model.predict(x_valid)
rmse_valid = mean_squared_error(y_valid, y_pred_valid)
print(f"RMSE (validação): {rmse_valid:.4f}")

y_pred_test = model.predict(x_test)
pearson_list = []
for i in range(y_test.shape[0]):
    corr_val = pearsonr(y_test[i], y_pred_test[i])[0]
    pearson_list.append(corr_val)

df_rec = pd.DataFrame()
df_rec['X2_pred_pearson'] = pearson_list
df_rec['cp_id'] = cp_id_test
df_rec['cid'] = cid_test
df_rec['sig'] = sig_test

smi_ls = [idx2smi[smi_id] for smi_id in df_rec['cp_id']]
df_rec['canonical_smiles'] = smi_ls

rmse_test = mean_squared_error(y_test, y_pred_test)
print(f"RMSE (teste): {rmse_test:.4f}")

print("\nExemplo de df_rec:")


RMSE (validação): 0.4772
RMSE (teste): 0.5383

Exemplo de df_rec:
    X2_pred_pearson  cp_id   cid      sig  \
0          0.951134      6  A375   A375_6   
1          0.950750      6  A549   A549_6   
2          0.958473      6  HA1E   HA1E_6   
3          0.887604      6  HT29   HT29_6   
4          0.973354      6  MCF7   MCF7_6   
5          0.979532      6   PC3    PC3_6   
6          0.975585      6  VCAP   VCAP_6   
7          0.926834      9  A375   A375_9   
8          0.927456      9  A549   A549_9   
9          0.953993      9  HA1E   HA1E_9   
10         0.877065      9  HT29   HT29_9   
11         0.936419      9  MCF7   MCF7_9   
12         0.952327      9   PC3    PC3_9   
13         0.948653      9  VCAP   VCAP_9   
14         0.957124     13  A549  A549_13   
15         0.961947     13   ASC   ASC_13   
16         0.977777     13  HA1E  HA1E_13   

                                     canonical_smiles  
0                   Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1  
1          

In [150]:
df_rec

,X2_pred_pearson,cp_id,cid,sig,canonical_smiles
0,0.951134,6,A375,A375_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
1,0.950750,6,A549,A549_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
2,0.958473,6,HA1E,HA1E_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
3,0.887604,6,HT29,HT29_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
4,0.973354,6,MCF7,MCF7_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
5,0.979532,6,PC3,PC3_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
6,0.975585,6,VCAP,VCAP_6,Brc1c[nH]c2nc(SCc3ccccc3C#N)nc2c1
7,0.926834,9,A375,A375_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
8,0.927456,9,A549,A549_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
9,0.953993,9,HA1E,HA1E_9,Brc1ccc(CSc2nnc(c3ccccn3)n2Cc4ccco4)cc1
